In [1]:
import pandas as pd
import time

# Pastikan nama file ini sesuai dengan dataset 6GB Anda yang ada di folder Skripsi_FDI
file_path = 'NF-ToN-IoT-v3.csv' 

print("Mulai membaca 1000 baris pertama...")
waktu_mulai = time.time()

# Kita batasi 1000 baris dan gunakan engine C agar secepat kilat
df_sample = pd.read_csv(file_path, nrows=1000, engine='c')

waktu_selesai = time.time()
print(f"✅ Selesai dalam {waktu_selesai - waktu_mulai:.2f} detik!")
print(f"Ukuran sampel: {df_sample.shape[0]} baris, {df_sample.shape[1]} kolom.")

# Tampilkan 5 baris pertama
display(df_sample.head())

Mulai membaca 1000 baris pertama...
✅ Selesai dalam 0.02 detik!
Ukuran sampel: 1000 baris, 55 kolom.


,FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,...,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
0,1556028590618,1556028590618,192.168.1.32,41315,192.168.1.169,10010,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
1,1556028590618,1556028590621,192.168.1.32,41315,192.168.1.194,49400,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
2,1556028590618,1556028590621,192.168.1.32,41315,192.168.1.193,8008,6,161,44,1,...,0,0,0,0,0,0,0,0,1,scanning
3,1556028590624,1556028590628,192.168.1.32,41315,192.168.1.194,85,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign
4,1556028590624,1556028590624,192.168.1.32,41317,192.168.1.180,1056,6,0,44,1,...,0,0,0,0,0,0,0,0,0,Benign


In [2]:
# 1. Mengecek nama kolom target (biasanya bernama 'Label' atau 'Attack')
print("Daftar 5 Kolom Terakhir (untuk mencari kolom Target/Label):")
print(df_sample.columns[-5:].tolist())
print("-" * 50)

# 2. Mengecek apakah ada nilai kosong (NaN) di dalam sampel
jumlah_nan = df_sample.isna().sum().sum()
print(f"Jumlah total nilai kosong (NaN) di sampel ini: {jumlah_nan}")

# 3. Mengecek tipe data untuk memastikan semuanya numerik (syarat masuk model)
tipe_non_numerik = df_sample.select_dtypes(exclude=['int64', 'float64']).columns.tolist()
print(f"Kolom dengan tipe data non-numerik (teks/kategori): {tipe_non_numerik}")

Daftar 5 Kolom Terakhir (untuk mencari kolom Target/Label):
['DST_TO_SRC_IAT_MAX', 'DST_TO_SRC_IAT_AVG', 'DST_TO_SRC_IAT_STDDEV', 'Label', 'Attack']
--------------------------------------------------
Jumlah total nilai kosong (NaN) di sampel ini: 0
Kolom dengan tipe data non-numerik (teks/kategori): ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Attack']


In [3]:
# Mari kita lihat daftar lengkap isi kolom 'Attack' di sampel ini
print("Daftar jenis kelas (Normal & Serangan) beserta jumlahnya:")
print(df_sample['Attack'].value_counts())

Daftar jenis kelas (Normal & Serangan) beserta jumlahnya:
Attack
Benign      967
scanning     33
Name: count, dtype: int64


In [4]:
file_path = 'NF-ToN-IoT-v3.csv' # Jangan lupa sesuaikan namanya

unique_attacks = set()
print("Sedang memindai seluruh file 6 GB secara bertahap (chunking)...")
print("Ini mungkin memakan waktu 1-3 menit, silakan ditunggu.")

# Kita hanya memuat kolom 'Attack' saja agar sangat ringan dan cepat
for chunk in pd.read_csv(file_path, chunksize=500000, usecols=['Attack'], engine='c'):
    # Tambahkan jenis serangan yang baru ditemukan ke dalam himpunan (set)
    unique_attacks.update(chunk['Attack'].dropna().unique())

print("\n✅ Pemindaian Selesai!")
print("Daftar LENGKAP semua jenis kelas/serangan di dataset ini:")
for attack in unique_attacks:
    print(f"- {attack}")

Sedang memindai seluruh file 6 GB secara bertahap (chunking)...
Ini mungkin memakan waktu 1-3 menit, silakan ditunggu.

✅ Pemindaian Selesai!
Daftar LENGKAP semua jenis kelas/serangan di dataset ini:
- password
- xss
- scanning
- mitm
- Backdoor
- injection
- ddos
- Benign
- ransomware
- dos


In [5]:
# Nama file 6 GB Anda (sesuaikan jika berbeda)
file_path = 'NF-ToN-IoT-v3.csv' 
# Nama file BARU yang akan jauh lebih kecil dan ringan
new_file_path = 'dataset_fdi_bersih.csv' 

# Kolom teks/kategori yang sepakat kita buang agar algoritma bisa belajar dengan baik
kolom_dibuang = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Attack']

print("Memulai proses penyaringan khusus kelas 'Benign' dan 'injection'...")
print("Sedang memproses dan membuat file baru (harap tunggu 1-3 menit)...")

header_ditulis = False
total_baris_tersimpan = 0

# Kita baca lagi 6 GB sepotong-sepotong agar RAM tetap aman
for chunk in pd.read_csv(file_path, chunksize=500000, engine='c'):
    
    # 1. Filter: Ambil hanya baris yang Attack-nya 'Benign' atau 'injection'
    chunk_filtered = chunk[chunk['Attack'].isin(['Benign', 'injection'])].copy()
    
    # Jika di potongan ini ada datanya, kita proses dan simpan
    if not chunk_filtered.empty:
        # 2. Buang kolom IP dan Attack (karena kita akan pakai kolom 'Label' 0/1 untuk targetnya)
        chunk_filtered = chunk_filtered.drop(columns=kolom_dibuang, errors='ignore')
        
        # 3. Simpan langsung ke file CSV baru
        if not header_ditulis:
            # Tulis file baru beserta nama kolomnya (header)
            chunk_filtered.to_csv(new_file_path, index=False, mode='w')
            header_ditulis = True
        else:
            # Tambahkan (append) ke baris bawahnya tanpa menulis nama kolom lagi
            chunk_filtered.to_csv(new_file_path, index=False, mode='a', header=False)
            
        total_baris_tersimpan += len(chunk_filtered)

print(f"\n✅ BINGO! Dataset skripsi berhasil dibuat: {new_file_path}")
print(f"Total data Anda sekarang mengerucut menjadi: {total_baris_tersimpan} baris.")

Memulai proses penyaringan khusus kelas 'Benign' dan 'injection'...
Sedang memproses dan membuat file baru (harap tunggu 1-3 menit)...

✅ BINGO! Dataset skripsi berhasil dibuat: dataset_fdi_bersih.csv
Total data Anda sekarang mengerucut menjadi: 17173991 baris.


In [6]:
new_file_path = 'dataset_fdi_bersih.csv'
print("Mengecek proporsi data Normal (0) vs Injection (1)...")

# Kita siapkan wadah kosong untuk menghitung
distribusi = pd.Series(dtype=int)

# Kita baca lagi pakai chunking agar RAM tetap aman
for chunk in pd.read_csv(new_file_path, chunksize=1000000, usecols=['Label'], engine='c'):
    # Tambahkan jumlah 0 dan 1 dari setiap potongan data
    distribusi = distribusi.add(chunk['Label'].value_counts(), fill_value=0)

print("\n✅ Hasil Distribusi Kelas pada Dataset Skripsi Anda:")
print(distribusi.astype(int))
print("-" * 50)
persentase_serangan = (distribusi.get(1, 0) / distribusi.sum()) * 100
print(f"Persentase Serangan Injection: {persentase_serangan:.2f}% dari total lalu lintas jaringan.")

Mengecek proporsi data Normal (0) vs Injection (1)...

✅ Hasil Distribusi Kelas pada Dataset Skripsi Anda:
Label
0    16792214
1      381777
dtype: int64
--------------------------------------------------
Persentase Serangan Injection: 2.22% dari total lalu lintas jaringan.


In [7]:
from river import compose, preprocessing, tree, metrics, stream

# 1. Membangun Arsitektur Pipeline Model
model = compose.Pipeline(
    preprocessing.MinMaxScaler(),
    tree.HoeffdingTreeClassifier()
)

# 2. Menyiapkan Metrik Evaluasi (Cocok untuk data timpang)
metric_bacc = metrics.BalancedAccuracy()
metric_recall = metrics.Recall() # Fokus melihat seberapa banyak serangan yang berhasil ditangkap

file_path = 'dataset_fdi_bersih.csv'
print("🚀 Memulai simulasi aliran data (Data Stream) dengan Incremental Learning...")

# Kita muat 50.000 baris pertama sebagai tes awal
df_test = pd.read_csv(file_path, nrows=50000, engine='c')
X = df_test.drop(columns=['Label'])
y = df_test['Label']

# 3. Looping Simulasi Aliran Data Sensor (Satu per satu)
for i, (x, y_true) in enumerate(stream.iter_pandas(X, y)):
    
    # a. Fase Uji: Model mencoba menebak data yang baru masuk
    y_pred = model.predict_one(x)
    
    # b. Fase Evaluasi: Mencatat apakah tebakan model benar atau salah
    if y_pred is not None:
        metric_bacc.update(y_true, y_pred)
        metric_recall.update(y_true, y_pred)
        
    # c. Fase Belajar: Model dikoreksi dan belajar dari data tersebut
    model.learn_one(x, y_true)
    
    # d. Laporan berkala setiap 10.000 data
    if (i + 1) % 10000 == 0:
        print(f"Mempelajari Paket Data ke-{i+1} | Balanced Acc: {metric_bacc.get():.4f} | Recall: {metric_recall.get():.4f}")

print("\n✅ Simulasi tes awal selesai!")
print(f"Performa Akhir Model pada 50k data pertama:")
print(f"- Balanced Accuracy : {metric_bacc.get() * 100:.2f}%")
print(f"- Recall (Deteksi)  : {metric_recall.get() * 100:.2f}%")

🚀 Memulai simulasi aliran data (Data Stream) dengan Incremental Learning...
Mempelajari Paket Data ke-10000 | Balanced Acc: 1.0000 | Recall: 0.0000
Mempelajari Paket Data ke-20000 | Balanced Acc: 1.0000 | Recall: 0.0000
Mempelajari Paket Data ke-30000 | Balanced Acc: 1.0000 | Recall: 0.0000
Mempelajari Paket Data ke-40000 | Balanced Acc: 1.0000 | Recall: 0.0000
Mempelajari Paket Data ke-50000 | Balanced Acc: 1.0000 | Recall: 0.0000

✅ Simulasi tes awal selesai!
Performa Akhir Model pada 50k data pertama:
- Balanced Accuracy : 100.00%
- Recall (Deteksi)  : 0.00%


In [8]:
# Cek isi 50.000 baris pertama
jumlah_serangan_awal = df_test['Label'].sum()
print(f"Jumlah serangan (Label 1) di 50.000 data pertama: {jumlah_serangan_awal} baris")

Jumlah serangan (Label 1) di 50.000 data pertama: 0 baris


In [ ]:
import pandas as pd
from river import compose, preprocessing, tree, metrics, stream
import time

# 1. Arsitektur Inti (Scaler dinamis + Hoeffding Tree)
model = compose.Pipeline(
    preprocessing.MinMaxScaler(),
    tree.HoeffdingTreeClassifier()
)

# 2. Metrik Pengukuran (Sangat sensitif terhadap data timpang)
metric_bacc = metrics.BalancedAccuracy()
metric_recall = metrics.Recall()

file_path = 'dataset_fdi_bersih.csv'
print("🚀 Memulai Simulasi Skala Penuh (17,1 Juta Data)...")
print("Ini akan memakan waktu (bisa belasan/puluhan menit), tapi RAM Anda aman.")

waktu_mulai = time.time()
total_diproses = 0

# 3. Membaca sepotong demi sepotong (500.000 baris) dari hard drive
for chunk in pd.read_csv(file_path, chunksize=500000, engine='c'):
    X_chunk = chunk.drop(columns=['Label'])
    y_chunk = chunk['Label']
    
    # 4. Aliran Data Real-Time: Masuk ke model SATU PER SATU
    for x, y_true in stream.iter_pandas(X_chunk, y_chunk):
        total_diproses += 1
        
        # a. Tebak dulu
        y_pred = model.predict_one(x)
        
        # b. Catat kebenarannya
        if y_pred is not None:
            metric_bacc.update(y_true, y_pred)
            metric_recall.update(y_true, y_pred)
            
        # c. Belajar dari jawaban yang benar
        model.learn_one(x, y_true)
        
        # 5. Cetak laporan proges setiap 1 Juta data agar terminal tidak penuh
        if total_diproses % 1000000 == 0:
            print(f"[{total_diproses:,} data] Balanced Acc: {metric_bacc.get():.4f} | Recall: {metric_recall.get():.4f}")

waktu_selesai = time.time()
print("\n✅ SIMULASI SKALA PENUH SELESAI!")
print(f"Waktu total eksekusi: {(waktu_selesai - waktu_mulai)/60:.2f} menit.")
print(f"=== PERFORMA AKHIR HOEFFDING TREE ===")
print(f"Total Data Terproses : {total_diproses:,} paket")
print(f"Balanced Accuracy    : {metric_bacc.get() * 100:.2f}%")
print(f"Recall (Deteksi FDI) : {metric_recall.get() * 100:.2f}%")